In [26]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch

from countcv.core.data import SyncedTransform
from countcv.core.densitymap_regression import _infer_full_image
from countcv.core.fcrn_model import SAUnet


In [27]:
def inference_on_path(path: Path, model: torch.nn.Module, device, transform_kwargs) -> torch.Tensor:
	transform = SyncedTransform(**transform_kwargs)

	image = cv2.imread(str(path))
	# Handle grayscale and color images
	if len(image.shape) == 2:  # Grayscale
		image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
	elif image.shape[2] == 4:  # RGBA
		image = cv2.cvtColor(image, cv2.COLOR_BGRA2RGB)
	else:  # BGR
		image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

	transformed_img = transform(image, mode="test", dot_annotations=None)
	transformed_img = transformed_img.unsqueeze(0).to(device)
	result = _infer_full_image(
		model,
		imgs=transformed_img,
		targets=torch.zeros_like(transformed_img),
		tile_size=getattr(transform_kwargs, "tilesize", None),
		device=device,
	)[0].squeeze()
	return transformed_img.squeeze().permute(1, 2, 0).cpu().numpy(), result.cpu().numpy()

In [28]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = Path("../checkpoints/SAUnet.pt")
img_path = "../data/uc_cells/images/test/011227.32-32.RB.TIF"
transform_kwargs = {
	"tilesize": 224,
	"centercrop": 220,
	"size": 220,
	"mean": (0.002109671, 0.0, 0.01138233),
	"std": (0.02599067, 1e-06, 0.066873804),
}
model = SAUnet(input_channels=3)
model.load_state_dict(torch.load(model_path))
model.to(device)
model.eval()

image, result = inference_on_path(img_path, model, device, transform_kwargs)
fig, axs = plt.subplots(ncols=2, figsize=(15, 15))

axs[0].imshow(image, vmin=-0.12, vmax=20)
axs[1].imshow(result, cmap="viridis")

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 54 but got size 55 for tensor number 1 in the list.